In [43]:
import pandas as pd

df = pd.read_csv(r"C:\Users\jeni7\OneDrive\Documents\CTS_Hackathon\CTS_Hackathon\hsp_df_processed.csv")
print("Dataset loaded successfully!")
print(df.head())


Dataset loaded successfully!
   time_in_hospital  n_lab_procedures  n_procedures  n_medications  \
0                 8                72             1             18   
1                 3                34             2             13   
2                 5                45             0             18   
3                 2                36             0             12   
4                 1                42             0              7   

   n_outpatient  n_inpatient  n_emergency  medical_specialty  Circulatory  \
0             2            0            0                  4            1   
1             0            0            0                  5            0   
2             0            0            0                  4            1   
3             1            0            0                  4            1   
4             0            0            0                  3            1   

   Respiratory  ...  Musculoskeletal  glucose_test  HbA1ctest  med_change  \
0         

In [44]:
# Numeric: median
X_train[numeric_cols] = X_train[numeric_cols].fillna(X_train[numeric_cols].median())
X_test[numeric_cols] = X_test[numeric_cols].fillna(X_train[numeric_cols].median())

# Categorical: mode (only if categorical columns exist)
if len(categorical_cols) > 0:
    X_train[categorical_cols] = X_train[categorical_cols].fillna(X_train[categorical_cols].mode().iloc[0])
    X_test[categorical_cols] = X_test[categorical_cols].fillna(X_train[categorical_cols].mode().iloc[0])


In [46]:
# -------------------------------
# Handle missing values
# -------------------------------
# Numeric: fill with median
X_train[numeric_cols] = X_train[numeric_cols].fillna(X_train[numeric_cols].median())
X_test[numeric_cols] = X_test[numeric_cols].fillna(X_train[numeric_cols].median())

# Categorical: only if they exist
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
if len(categorical_cols) > 0:
    # Only fill if there are columns
    mode_values = X_train[categorical_cols].mode()
    if not mode_values.empty:
        X_train[categorical_cols] = X_train[categorical_cols].fillna(mode_values.iloc[0])
        X_test[categorical_cols] = X_test[categorical_cols].fillna(mode_values.iloc[0])


In [49]:
# -------------------------------
# Handle missing values
# -------------------------------
# Numeric: fill missing values with median
X_train[numeric_cols] = X_train[numeric_cols].fillna(X_train[numeric_cols].median())
X_test[numeric_cols] = X_test[numeric_cols].fillna(X_train[numeric_cols].median())

# Categorical: skip if no categorical columns
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
if len(categorical_cols) > 0:
    # Only fill if categorical columns exist
    mode_values = X_train[categorical_cols].mode()
    if not mode_values.empty:
        X_train[categorical_cols] = X_train[categorical_cols].fillna(mode_values.iloc[0])
        X_test[categorical_cols] = X_test[categorical_cols].fillna(mode_values.iloc[0])


In [51]:
# Numeric columns: fill missing values with median
X_train[numeric_cols] = X_train[numeric_cols].fillna(X_train[numeric_cols].median())
X_test[numeric_cols] = X_test[numeric_cols].fillna(X_train[numeric_cols].median())

# Categorical columns: only if they exist
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
if categorical_cols:   # only enter block if list is not empty
    mode_values = X_train[categorical_cols].mode()
    if not mode_values.empty:
        X_train[categorical_cols] = X_train[categorical_cols].fillna(mode_values.iloc[0])
        X_test[categorical_cols] = X_test[categorical_cols].fillna(mode_values.iloc[0])


In [53]:
# -------------------------------
# Handle missing values
# -------------------------------
# Numeric columns: fill missing values with median
X_train[numeric_cols] = X_train[numeric_cols].fillna(X_train[numeric_cols].median())
X_test[numeric_cols] = X_test[numeric_cols].fillna(X_train[numeric_cols].median())

# Skip categorical columns completely (none exist in this dataset)


In [55]:
# -------------------------------
# 1️⃣ Import libraries
# -------------------------------
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report

# -------------------------------
# 2️⃣ Load dataset
# -------------------------------
df = pd.read_csv(r"C:\Users\jeni7\OneDrive\Documents\CTS_Hackathon\CTS_Hackathon\hsp_df_processed.csv")
print("Dataset loaded successfully!")
print(df.head())

# -------------------------------
# 3️⃣ Split features & target
# -------------------------------
X = df.drop("readmitted", axis=1)
y = df["readmitted"]

# -------------------------------
# 4️⃣ Train-test split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------------
# 5️⃣ Identify numeric columns
# -------------------------------
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# -------------------------------
# 6️⃣ Handle missing values (numeric only)
# -------------------------------
X_train[numeric_cols] = X_train[numeric_cols].fillna(X_train[numeric_cols].median())
X_test[numeric_cols] = X_test[numeric_cols].fillna(X_train[numeric_cols].median())

# -------------------------------
# 7️⃣ Scale numeric features
# -------------------------------
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

# -------------------------------
# 8️⃣ Balance classes using SMOTE
# -------------------------------
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

# -------------------------------
# 9️⃣ Train CatBoost model
# -------------------------------
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric='Accuracy',
    random_seed=42,
    verbose=100
)

model.fit(X_train_res, y_train_res, eval_set=(X_test_scaled, y_test))

# -------------------------------
# 🔟 Evaluate model
# -------------------------------
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", round(accuracy*100, 2), "%")
print(classification_report(y_test, y_pred))


Dataset loaded successfully!
   time_in_hospital  n_lab_procedures  n_procedures  n_medications  \
0                 8                72             1             18   
1                 3                34             2             13   
2                 5                45             0             18   
3                 2                36             0             12   
4                 1                42             0              7   

   n_outpatient  n_inpatient  n_emergency  medical_specialty  Circulatory  \
0             2            0            0                  4            1   
1             0            0            0                  5            0   
2             0            0            0                  4            1   
3             1            0            0                  4            1   
4             0            0            0                  3            1   

   Respiratory  ...  Musculoskeletal  glucose_test  HbA1ctest  med_change  \
0         

In [57]:
# Feature Engineering: only if columns exist
if 'age' in df.columns:
    df['age_bin'] = pd.cut(df['age'], bins=[0,30,50,70,120], labels=[0,1,2,3])

if 'n_medications' in df.columns and 'n_procedures' in df.columns:
    df['med_per_procedure'] = df['n_medications'] / (df['n_procedures'] + 1)

if all(c in df.columns for c in ['n_lab_procedures','n_procedures','n_medications']):
    df['stay_intensity'] = df['n_lab_procedures'] + df['n_procedures'] + df['n_medications']


In [58]:
# -------------------------------
# 1️⃣ Import libraries
# -------------------------------
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report

# -------------------------------
# 2️⃣ Load dataset
# -------------------------------
df = pd.read_csv(r"C:\Users\jeni7\OneDrive\Documents\CTS_Hackathon\CTS_Hackathon\hsp_df_processed.csv")
print("Dataset loaded successfully!")
print(df.head())

# -------------------------------
# 3️⃣ Feature Engineering (safe)
# -------------------------------
if 'age' in df.columns:
    df['age_bin'] = pd.cut(df['age'], bins=[0,30,50,70,120], labels=[0,1,2,3])

if 'n_medications' in df.columns and 'n_procedures' in df.columns:
    df['med_per_procedure'] = df['n_medications'] / (df['n_procedures'] + 1)

if all(c in df.columns for c in ['n_lab_procedures','n_procedures','n_medications']):
    df['stay_intensity'] = df['n_lab_procedures'] + df['n_procedures'] + df['n_medications']

# -------------------------------
# 4️⃣ Split features & target
# -------------------------------
X = df.drop("readmitted", axis=1)
y = df["readmitted"]

# -------------------------------
# 5️⃣ Train-test split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------------
# 6️⃣ Numeric columns
# -------------------------------
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# -------------------------------
# 7️⃣ Handle missing values (numeric only)
# -------------------------------
X_train[numeric_cols] = X_train[numeric_cols].fillna(X_train[numeric_cols].median())
X_test[numeric_cols] = X_test[numeric_cols].fillna(X_train[numeric_cols].median())

# -------------------------------
# 8️⃣ Scale numeric features
# -------------------------------
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

# -------------------------------
# 9️⃣ Balance classes using SMOTE
# -------------------------------
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

# -------------------------------
# 🔟 Train CatBoost model
# -------------------------------
model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=5,
    eval_metric='Accuracy',
    random_seed=42,
    verbose=100
)

model.fit(X_train_res, y_train_res, eval_set=(X_test_scaled, y_test))

# -------------------------------
# 1️⃣1️⃣ Evaluate model
# -------------------------------
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", round(accuracy*100,2), "%")
print(classification_report(y_test, y_pred))


Dataset loaded successfully!
   time_in_hospital  n_lab_procedures  n_procedures  n_medications  \
0                 8                72             1             18   
1                 3                34             2             13   
2                 5                45             0             18   
3                 2                36             0             12   
4                 1                42             0              7   

   n_outpatient  n_inpatient  n_emergency  medical_specialty  Circulatory  \
0             2            0            0                  4            1   
1             0            0            0                  5            0   
2             0            0            0                  4            1   
3             1            0            0                  4            1   
4             0            0            0                  3            1   

   Respiratory  ...  Musculoskeletal  glucose_test  HbA1ctest  med_change  \
0         